# Multi-Class Sentiment Analyzer with Error Analysis

This notebook implements the basic NLP sentiment-analysis pipeline specified in the provided Markdown assignment, adapted to the customer-feedback dataset used in this project.

The model predicts three classes: **positive, neutral, negative**.

**Methodology:** Dataset → Text Cleaning → Train-Test Split → TF-IDF → Logistic Regression → Prediction → Evaluation → Confusion Matrix → Error Analysis

## 1. Import Required Libraries

The assignment specifies pandas, numpy, matplotlib, and scikit-learn.

- pandas: dataset handling
- numpy: numerical operations
- re: text cleaning
- matplotlib: visualization
- scikit-learn: splitting, TF-IDF, Logistic Regression, and evaluation

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## 2. Load the Dataset

Our project dataset is `../data/feedback.csv`. It contains `feedback`, `sentiment`, and `category`.

For this notebook, `feedback` is the input and `sentiment` is the target. The category column is intentionally not used here because category classification is handled separately.

In [ ]:
DATA_PATH = "../data/feedback.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Check Dataset Information

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

## 4. Remove Missing Values

In [ ]:
df = df.dropna(subset=["feedback", "sentiment"]).copy()
print("Shape after removing missing values:", df.shape)
print(df[["feedback", "sentiment"]].head())

## 5. Check Sentiment Classes

This is **multi-class classification** because the model predicts three possible classes: positive, neutral, and negative.

In [ ]:
print("Sentiment classes:")
print(df["sentiment"].unique())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

## 6. Visualize Sentiment Distribution

In [ ]:
df["sentiment"].value_counts().plot(
    kind="bar",
    title="Sentiment Distribution"
)
plt.xlabel("Sentiment")
plt.ylabel("Number of Samples")
plt.show()

## 7. Text Preprocessing

Following the assignment, cleaning performs four basic operations:

1. lowercase text
2. remove URLs
3. remove non-letter characters
4. remove extra spaces

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


df["clean_text"] = df["feedback"].apply(clean_text)
display(df[["feedback", "clean_text"]].head())

## 8. Define Input and Target

In [ ]:
X = df["clean_text"]
y = df["sentiment"]

print("Input:")
display(X.head())
print("Target:")
display(y.head())

## 9. Train-Test Split

The assignment uses an 80-20 split. `stratify=y` maintains approximately the same sentiment proportions in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 10. TF-IDF Feature Extraction

The assignment specifies `max_features=5000` and `ngram_range=(1, 2)`. This lets the model use both single words and two-word phrases.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

## 11. Why `fit_transform()` for Training and `transform()` for Testing?

The vectorizer learns vocabulary and IDF information from training data only. The test set is transformed using that learned representation, preventing test-set information from influencing training.

## 12. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("Model training completed.")

## 13. Make Predictions

In [ ]:
y_pred = model.predict(X_test_tfidf)
print("First 20 predictions:")
print(y_pred[:20])

## 14. Accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy is the proportion of test predictions that are correct. The actual value must be taken from the executed notebook; no fixed accuracy is assumed.

## 15. Classification Report

In [ ]:
print(classification_report(y_test, y_pred))

The classification report provides precision, recall, F1-score, and support for each class.

## 16. Confusion Matrix

In [ ]:
labels = ["negative", "neutral", "positive"]
cm = confusion_matrix(y_test, y_pred, labels=labels)

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)
display_cm.plot()
plt.title("Sentiment Confusion Matrix")
plt.show()

The diagonal contains correct predictions. Values outside the diagonal represent errors, such as neutral being predicted as positive.

## 17. Error Analysis

In [ ]:
results = pd.DataFrame({
    "text": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred
})

errors = results[results["actual"] != results["predicted"]].copy()

print("Total test samples:", len(results))
print("Incorrect predictions:", len(errors))
display(errors.head(20))

## 18. Error Counts by Actual and Predicted Class

In [ ]:
error_counts = errors.groupby(["actual", "predicted"]).size()
print(error_counts)

## 19. Common Error Patterns

The supplied assignment highlights these weaknesses of a basic TF-IDF + Logistic Regression model:

- **Negation:** `not bad` may be confused because the model sees `bad`.
- **Mixed opinions:** one sentence can contain positive and negative signals.
- **Neutral sentiment:** neutral language often has weak emotional cues.
- **Sarcasm:** positive words can express a negative meaning.
- **Rare vocabulary:** uncommon words may not have been learned well.
- **Context:** TF-IDF does not deeply understand sentence meaning.

## 20. Inspect Neutral-Sentiment Errors

In [ ]:
neutral_errors = errors[errors["actual"] == "neutral"]
print("Neutral examples incorrectly classified:")
display(neutral_errors.head(10))

## 21. Error Rate

In [ ]:
error_rate = len(errors) / len(results)
print("Error rate:", error_rate)

## 22. Predict New Customer Feedback

In [ ]:
def predict_sentiment(sentence):
    cleaned = clean_text(sentence)
    sentence_tfidf = tfidf.transform([cleaned])
    return model.predict(sentence_tfidf)[0]

In [ ]:
test_sentences = [
    "The payment worked perfectly and I am very happy.",
    "The application is slow and keeps failing.",
    "I want information about my recent order."
]

for sentence in test_sentences:
    print(sentence, "->", predict_sentiment(sentence))

## 23. Final Methodology

```text
Dataset
   ↓
Text Cleaning
   ↓
Train-Test Split
   ↓
TF-IDF Vectorization
   ↓
Logistic Regression
   ↓
Sentiment Prediction
   ↓
Model Evaluation
   ↓
Confusion Matrix
   ↓
Error Analysis
```

This notebook intentionally follows the basic NLP assignment. Transformer/BERT experiments belong in `05_transformers.ipynb`, and multi-label category prediction belongs in the multi-label notebook.

## 24. Conclusion

The notebook builds a three-class sentiment analyzer for customer feedback using text cleaning, TF-IDF, and Logistic Regression. It evaluates the model with accuracy, precision, recall, F1-score, and a confusion matrix, then examines incorrect predictions through error analysis.

The actual metrics should be taken from the executed notebook.